# 用 Groq 把博客网站摘要成 LinkedIn 帖子

## 练习目标（理念）

把「网页抓取 + LLM 摘要」接到 **Groq**（OpenAI 兼容接口）上：输入博客 URL，输出适合发 LinkedIn 的段落式摘要。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat / Responses API | `client.responses.create(...)` |
| `messages`（system / user） | system 定「LinkedIn 风格」，user 放博客正文 |
| 第三方兼容端点 | `base_url=https://api.groq.com/openai/v1` |
| 网页正文抓取 | `fetch_website_contents(url)`（来自 `scraper`） |

## 怎么跑

1. 准备 `.env`：至少有 `GROQ_API_KEY`
2. 确保同目录可用 `scraper.fetch_website_contents`
3. 从上到下运行单元格；在最后一格改 URL 试其他博客


In [1]:
# ========== 导入 + Groq 客户端（OpenAI 兼容） ==========

# 从 openai 导入 OpenAI 客户端类：也可指向 Groq 等兼容端点
from openai import OpenAI
# 导入标准库 os：从环境变量读取 GROQ_API_KEY
import os
# 从本地 scraper 导入 fetch_website_contents：抓取并返回网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown / display：本练习可选用，用于漂亮展示
from IPython.display import Markdown, display
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv

# 加载 .env；override=True 表示已有同名环境变量也会被 .env 覆盖
load_dotenv(override=True)

# 创建客户端：密钥走 GROQ_API_KEY；base_url 指向 Groq 的 OpenAI 兼容 /v1
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)


In [2]:
# ========== system prompt：定「博客 → LinkedIn 帖」角色 ==========

# 定义 system 提示——可自行实验（例如改成西班牙语 Markdown）
# 注意：影响模型行为的英文 prompt 保持原样，不要翻译字符串本身

system_prompt = """
You are a expert in summarizing the contents of a blog website,
and provides a short summary of the blog website  , ignoring text that might be navigation related.
Respond in paragraph format and in such a way that it is ready for linkedin post.
"""


In [3]:
# ========== user prompt 前缀：后面会拼上抓取到的博客正文 ==========

# 影响回答的英文提示保持原样；正文由 messages_for 拼到后面
user_prompt="""
Here are the contents of a blog website.
Provide a short summary of  this blog website.
Summarize in professional format.
"""


In [4]:
# ========== 组装 messages：system + user（user = 前缀 + 网页正文） ==========

def messages_for(website):
    """把 system_prompt 与「user 前缀 + 正文」打成 Chat Completions 风格列表。"""
    # 返回两条消息：角色分工清晰，便于 responses.create 的 input=
    return [
        {"role": "system", "content": system_prompt},
        # website 这里是抓取到的正文字符串（不是 Website 对象）
        {"role": "user", "content": user_prompt + website}
    ]


In [5]:
# ========== 主流程：抓取 → Groq Responses API → 返回帖子文本 ==========

def summarized(url):
    """对博客 URL 生成 LinkedIn 风格摘要字符串。"""
    # 抓取网页正文（依赖 scraper；失败时看同目录 scraper 实现）
    website = fetch_website_contents(url)
    # 调用 Groq 上的 Responses API（OpenAI 新风格）；model id 保持原样
    response = client.responses.create(
    input=messages_for(website),
    model="openai/gpt-oss-20b",
    )
    # output_text：把模型输出收成单个字符串，方便直接打印或再 display
    return response.output_text


In [6]:
# ========== 试跑：换 URL 即可测其他 AI/技术博客 ==========

# 对 DigitalOcean 的 AI blogs 资源页做摘要；返回值会显示在单元格 output 里
summarized("https://www.digitalocean.com/resources/articles/ai-blogs")


'**DigitalOcean’s “12 AI Blogs for Keeping Up With AI Trends in 2026”**  \nDigitalOcean’s latest AI‑centric blog post is a concise, hand‑picked guide to the most insightful AI blogs of 2026. The post serves as a go‑to resource for data scientists, ML engineers, and cloud architects who want to stay ahead of emerging AI developments without sifting through endless content. By highlighting 12 leading blogs—from industry research hubs to community‑run platforms—DigitalOcean provides readers with a clear roadmap to deepen their knowledge, discover best practices, and connect with thought leaders in AI, all while showcasing the company’s own AI‑powered services such as Gradient™ and 1‑Click Models. This curated list not only saves time but also positions DigitalOcean as a trusted curator of AI innovation.'